# Clase 10 - Capa SILVER - Limpieza y enriquecimiento

Toma los datos crudos de Bronze, los limpia, tipa correctamente y los enriquece con joins a las tablas maestras (clientes, productos, tiendas).

**Prerequisito:** haber ejecutado el notebook `01_Bronze_Ingesta`.

Resultado: una unica tabla Silver con todas las ventas + datos descriptivos, lista para analitica.

## Paso 1 - Setup (mismo Volume y catalogo que en Bronze)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date

spark = SparkSession.builder.appName("SilverTransformClase10").getOrCreate()

BASE_VOL = "/Volumes/dbx_diplo_ricardo/default/clase8/clase10"
print("Volume:", BASE_VOL)

## Paso 2 - Leer las 4 tablas Bronze (todo viene como string)

In [ ]:
ventas_df    = spark.read.format("delta").load(f"{BASE_VOL}/bronze/ventas_diarias")
clientes_df  = spark.read.format("delta").load(f"{BASE_VOL}/bronze/clientes")
productos_df = spark.read.format("delta").load(f"{BASE_VOL}/bronze/productos")
tiendas_df   = spark.read.format("delta").load(f"{BASE_VOL}/bronze/tiendas")

print("ventas bronze:",    ventas_df.count())
print("clientes bronze:",  clientes_df.count())
print("productos bronze:", productos_df.count())
print("tiendas bronze:",   tiendas_df.count())

ventas_df.printSchema()

## Paso 3 - Limpieza de ventas

1. `dropna()` -> elimina filas con cualquier campo nulo.
2. `dropDuplicates([...])` -> elimina duplicados por clave de negocio.
3. Casts: `fecha -> date`, `cantidad -> int`, `precio_unitario -> double`.

In [ ]:
ventas_clean_df = (ventas_df
                   .dropna()
                   .dropDuplicates(["fecha", "tienda_id", "producto_id", "cliente_id"])
                   .withColumn("fecha",           to_date(col("fecha"), "yyyy-MM-dd"))
                   .withColumn("cantidad",        col("cantidad").cast("int"))
                   .withColumn("precio_unitario", col("precio_unitario").cast("double"))
                  )

print("ventas tras limpieza:", ventas_clean_df.count())
ventas_clean_df.printSchema()
ventas_clean_df.show(5)

## Paso 4 - Limpieza minima de las maestras

- `clientes`: dropeamos `ciudad` para evitar colision con `ciudad` de `tiendas` en el join.
- Tambien `dropna` y `dropDuplicates` por las dudas.

In [ ]:
clientes_clean  = clientes_df.dropna().dropDuplicates().drop("ciudad")
productos_clean = productos_df.dropna().dropDuplicates()
tiendas_clean   = tiendas_df.dropna().dropDuplicates()

clientes_clean.printSchema()
productos_clean.printSchema()
tiendas_clean.printSchema()

## Paso 5 - Enriquecimiento via 3 inner joins

In [ ]:
ventas_enriched_df = (ventas_clean_df
                      .join(clientes_clean,  on="cliente_id",  how="inner")
                      .join(productos_clean, on="producto_id", how="inner")
                      .join(tiendas_clean,   on="tienda_id",   how="inner")
                     )

print("Filas Silver:", ventas_enriched_df.count())
ventas_enriched_df.printSchema()
ventas_enriched_df.show(5, truncate=False)

## Paso 6 - Escribir Silver en Delta y registrar en Unity Catalog

In [ ]:
silver_path = f"{BASE_VOL}/silver/ventas_enriched"

(ventas_enriched_df.write
                   .format("delta")
                   .mode("overwrite")
                   .option("overwriteSchema", "true")
                   .save(silver_path))

spark.sql("CREATE SCHEMA IF NOT EXISTS dbx_diplo_ricardo.silver_clase10")
spark.sql("DROP TABLE IF EXISTS dbx_diplo_ricardo.silver_clase10.silver_ventas")
spark.sql(f"""
    CREATE TABLE dbx_diplo_ricardo.silver_clase10.silver_ventas
    USING DELTA
    LOCATION '{silver_path}'
""")

print("Silver guardada en:", silver_path)
spark.sql("SHOW TABLES IN dbx_diplo_ricardo.silver_clase10").show(truncate=False)

## Paso 7 - Validacion: SQL sobre la tabla Silver

In [ ]:
%sql
SELECT fecha, nombre_tienda, ciudad, nombre_producto, categoria, cantidad, precio_unitario
FROM dbx_diplo_ricardo.silver_clase10.silver_ventas
ORDER BY fecha DESC
LIMIT 10;

In [ ]:
%sql
-- Conteo por categoria, util para sanity check
SELECT categoria, COUNT(*) AS ventas, SUM(cantidad) AS unidades
FROM dbx_diplo_ricardo.silver_clase10.silver_ventas
GROUP BY categoria
ORDER BY unidades DESC;